In [ ]:
# Notebook: Gibbs Phenomenon in a Causal Rectangular Pulse Fourier Reconstruction
# Author: Adapted for Springer-style presentation

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, BoundedIntText, FloatSlider


# ==========================
# Parameters
# ==========================

t = np.linspace(-2, 10, 2000)


# ==========================
# Causal Rectangular Pulse Signal
# ==========================

def x_causal_rect(t, A, T_pulse, T_period):
    t_mod = t % T_period
    return np.where((t_mod >= 0) & (t_mod <= T_pulse), A, 0.0)


# ==========================
# Fourier Expansion (Gibbs Phenomenon)
# ==========================

def x_fourier_causal_rect(t, N, A, T_pulse, T_period):
    omega = 2 * np.pi / T_period
    
    a0 = (A * T_pulse) / T_period
    xf = np.full_like(t, a0, dtype=float)
    
    for n in range(1, N + 1):
        an = (2 * A / (n * omega * T_period)) * np.sin(n * omega * T_pulse)
        bn = (2 * A / (n * omega * T_period)) * (1 - np.cos(n * omega * T_pulse))
        
        xf += an * np.cos(n * omega * t) + bn * np.sin(n * omega * t)
        
    return xf


# ==========================
# Interactive Plot Function
# ==========================

@interact(
    N=BoundedIntText(value=5, min=1, max=50, step=1, description="N"),
    A=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="A"),
    T_pulse=FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1, description="T (pulse)"),
    T_period=FloatSlider(value=6.0, min=2.0, max=10.0, step=0.1, description="Period")
)
def plot_gibbs(N, A, T_pulse, T_period):
    if T_pulse >= T_period:
        print("Σφάλμα: Η περίοδος πρέπει να είναι μεγαλύτερη από τη διάρκεια του παλμού.")
        return

    xx = x_causal_rect(t, A, T_pulse, T_period)
    xf = x_fourier_causal_rect(t, N, A, T_pulse, T_period)

    fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.plot(
        t, xx,
        'r',
        linewidth=3,
        label="Original causal pulse"
    )

    ax.plot(
        t, xf,
        'b',
        linewidth=2,
        label=f"Fourier expansion (N={N})"
    )

    ax.set_xlabel(r"$t$")
    ax.set_ylabel("Amplitude")
    ax.grid(True)
    
    # Υπόμνημα κάτω από τον άξονα x
    ax.legend(
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.18), 
        ncol=2, 
        frameon=True
    )

    ax.set_title(
        "Gibbs Phenomenon in Causal Rectangular Pulse Fourier Reconstruction"
    )
    
    fig.subplots_adjust(bottom=0.2)
    
    plt.show()
    plt.close(fig)